
# E-Ambulance — 02 Build OSMnx Network + Snap Points + OD Cache

Notebook này nối tiếp `01_EDA_Pre_OSMnx.ipynb`.

## Mục tiêu

1. Đọc `osmnx_points.csv` và `osmnx_bbox.json` từ bước EDA.
2. Tải **một** road graph OSM cho study area.
3. Gán `speed_kph` và `travel_time` lên edges.
4. Lưu graph thành `study_graph.graphml` để không tải lại mỗi lần.
5. Snap toàn bộ incident / fleet / hospital / charger vào OSM nodes bằng một lần gọi vectorized.
6. Kiểm tra `snap_distance_m`.
7. Sinh OD cache tĩnh:
   - `incident → hospital`;
   - `hospital → charger`.
8. Chuẩn bị framework cho routing động `vehicle → incident` trong rolling DES.

## Nguyên tắc

- Không route trực tiếp 540.000 candidate rows.
- Network layer chỉ tạo **free-flow baseline**.
- Traffic theo thời gian sẽ áp dụng sau theo dạng:

\[
T_{uv}(t)=T^{freeflow}_{uv}\times\gamma_{uv}(t,\omega)
\]

- `hospital capability`, `capacity`, `SOC`, `urgency` **không nằm trong OSMnx layer**. Chúng được join/filter ở decision layer.
- Emergency-vehicle privileges (đèn ưu tiên, luật đi đường đặc biệt, lane access...) **không được OSM `drive` mặc định mô hình hóa**.


In [1]:

from pathlib import Path
import json
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

# Tự tìm project root
cwd = Path.cwd().resolve()
ROOT = None
for p in [cwd, cwd.parent, *cwd.parents]:
    if (p / "data" / "sample").exists():
        ROOT = p
        break

if ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy project root có data/sample. "
        "Hãy mở notebook từ thư mục project hoặc sửa ROOT thủ công."
    )

EDA = ROOT / "eda_pre_osmnx"
DATA = ROOT / "data" / "sample"
OSM = ROOT / "data" / "osmnx"
OSM.mkdir(parents=True, exist_ok=True)

print("ROOT =", ROOT)
print("EDA  =", EDA)
print("OSM  =", OSM)


ROOT = /mnt/data/eambulance_dataset_v2_rebuilt
EDA  = /mnt/data/eambulance_dataset_v2_rebuilt/eda_pre_osmnx
OSM  = /mnt/data/eambulance_dataset_v2_rebuilt/data/osmnx


## 1. Cấu hình chạy

In [2]:

# Đặt True trên máy local khi sẵn sàng tải OSM từ Overpass.
RUN_NETWORK = False

# Nếu graph đã tồn tại, mặc định load từ disk thay vì tải lại.
FORCE_REDOWNLOAD = False

NETWORK_TYPE = "drive"

# QA threshold thực dụng, KHÔNG phải tham số khoa học cố định.
# Các point có snap distance lớn hơn threshold này cần review.
REVIEW_SNAP_DISTANCE_M = 250.0

# Fleet→incident initial OD không phải core static cache vì fleet location
# sẽ thay đổi theo policy trong rolling DES. Mặc định không build.
BUILD_INITIAL_FLEET_INCIDENT_OD = False

print({
    "RUN_NETWORK": RUN_NETWORK,
    "FORCE_REDOWNLOAD": FORCE_REDOWNLOAD,
    "NETWORK_TYPE": NETWORK_TYPE,
    "REVIEW_SNAP_DISTANCE_M": REVIEW_SNAP_DISTANCE_M,
    "BUILD_INITIAL_FLEET_INCIDENT_OD": BUILD_INITIAL_FLEET_INCIDENT_OD,
})


{'RUN_NETWORK': False, 'FORCE_REDOWNLOAD': False, 'NETWORK_TYPE': 'drive', 'REVIEW_SNAP_DISTANCE_M': 250.0, 'BUILD_INITIAL_FLEET_INCIDENT_OD': False}


## 2. Kiểm tra input từ Notebook 01

In [3]:

required = [
    EDA / "osmnx_points.csv",
    EDA / "osmnx_bbox.json",
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Thiếu output từ Notebook 01:\n" + "\n".join(missing)
    )

points = pd.read_csv(EDA / "osmnx_points.csv")
bbox_cfg = json.loads((EDA / "osmnx_bbox.json").read_text(encoding="utf-8"))

inc = pd.read_csv(DATA / "incidents.csv.gz")
hosp = pd.read_csv(DATA / "hospitals.csv")
chg = pd.read_csv(DATA / "charging_stations.csv")
fleet = pd.read_csv(DATA / "fleet_initial.csv.gz")

display(points.head())
print("points rows:", len(points))
print("unique coordinates:", points[["latitude", "longitude"]].drop_duplicates().shape[0])
print(json.dumps(bbox_cfg, indent=2))


,point_type,entity_id,scenario_id,replication_id,latitude,longitude
0,incident,NO_R001_I001,normal,R001,10.810727,106.656619
1,incident,NO_R001_I002,normal,R001,10.752264,106.691580
2,incident,NO_R001_I003,normal,R001,10.847279,106.650001
3,incident,NO_R001_I004,normal,R001,10.725125,106.641981
4,incident,NO_R001_I005,normal,R001,10.766829,106.745371


points rows: 9910
unique coordinates: 9910
{
  "raw": {
    "south": 10.677924,
    "north": 10.855636,
    "west": 106.555227,
    "east": 106.802642,
    "approx_north_south_km": 19.783,
    "approx_east_west_km": 27.057
  },
  "suggested_graph_download_bbox_with_2km_margin": {
    "south": 10.659958,
    "north": 10.873602,
    "west": 106.536939,
    "east": 106.82093,
    "note": "2 km ch\u1ec9 l\u00e0 download margin th\u1ef1c d\u1ee5ng; kh\u00f4ng ph\u1ea3i tham s\u1ed1 khoa h\u1ecdc c\u1ed1 \u0111\u1ecbnh."
  }
}



## 3. Cài OSMnx

Trên Terminal của môi trường `.venv`:

```bash
pip install "osmnx>=2.1,<3" scikit-learn
```

OSMnx 2.1.x dùng bbox theo thứ tự:

```text
(left, bottom, right, top)
= (west, south, east, north)
```

Notebook sẽ dừng phần network nếu `RUN_NETWORK=False`.


In [4]:

try:
    import osmnx as ox
    OSMNX_AVAILABLE = True
    print("OSMnx version:", ox.__version__)
except Exception as e:
    OSMNX_AVAILABLE = False
    ox = None
    print("OSMnx chưa sẵn sàng:", repr(e))

if RUN_NETWORK and not OSMNX_AVAILABLE:
    raise ImportError(
        'RUN_NETWORK=True nhưng chưa import được OSMnx. '
        'Chạy: pip install "osmnx>=2.1,<3" scikit-learn'
    )


OSMnx chưa sẵn sàng: ModuleNotFoundError("No module named 'osmnx'")


## 4. Chuẩn bị bbox

In [5]:

b = bbox_cfg["suggested_graph_download_bbox_with_2km_margin"]

WEST = float(b["west"])
SOUTH = float(b["south"])
EAST = float(b["east"])
NORTH = float(b["north"])

# OSMnx 2.1.x: (left, bottom, right, top)
BBOX = (WEST, SOUTH, EAST, NORTH)

bbox_table = pd.DataFrame([{
    "west": WEST,
    "south": SOUTH,
    "east": EAST,
    "north": NORTH,
    "network_type": NETWORK_TYPE,
}])

bbox_table


,west,south,east,north,network_type
0,106.536939,10.659958,106.82093,10.873602,drive



## 5. Download hoặc load road graph

Graph chỉ nên được tải một lần. Sau đó dùng `study_graph.graphml`.

`network_type="drive"` là baseline hợp lý cho road network thông thường, nhưng **không đại diện đầy đủ quyền ưu tiên của xe cấp cứu**.


In [6]:

GRAPH_PATH = OSM / "study_graph.graphml"

G = None
graph_source = None

if RUN_NETWORK:
    ox.settings.use_cache = True
    ox.settings.log_console = True

    if GRAPH_PATH.exists() and not FORCE_REDOWNLOAD:
        t0 = time.time()
        G = ox.io.load_graphml(GRAPH_PATH)
        graph_source = "graphml_cache"
        print(f"Loaded graph from {GRAPH_PATH} in {time.time()-t0:.2f}s")
    else:
        t0 = time.time()
        G = ox.graph.graph_from_bbox(
            BBOX,
            network_type=NETWORK_TYPE,
            simplify=True,
            retain_all=False,
            truncate_by_edge=True,
        )
        graph_source = "overpass_download"
        print(f"Downloaded graph in {time.time()-t0:.2f}s")

        # Free-flow speeds and travel times.
        # Đây mới là baseline; sau này cần local calibration.
        G = ox.routing.add_edge_speeds(G)
        G = ox.routing.add_edge_travel_times(G)

        ox.io.save_graphml(G, GRAPH_PATH)
        print("Saved:", GRAPH_PATH)

    # Nếu graph cache cũ chưa có speed/travel_time thì bổ sung.
    sample_edge = next(iter(G.edges(data=True)))[2]
    if "speed_kph" not in sample_edge:
        G = ox.routing.add_edge_speeds(G)
    if "travel_time" not in sample_edge:
        G = ox.routing.add_edge_travel_times(G)
        ox.io.save_graphml(G, GRAPH_PATH)

    print("nodes:", G.number_of_nodes())
    print("edges:", G.number_of_edges())
else:
    print("RUN_NETWORK=False → bỏ qua download/load graph.")


RUN_NETWORK=False → bỏ qua download/load graph.


## 6. Network metadata và QA edges

In [7]:

if G is not None:
    edge_rows = []
    for u, v, k, d in G.edges(keys=True, data=True):
        edge_rows.append({
            "u": u,
            "v": v,
            "key": k,
            "length_m": d.get("length"),
            "speed_kph": d.get("speed_kph"),
            "travel_time_s": d.get("travel_time"),
            "highway": d.get("highway"),
            "maxspeed_raw": d.get("maxspeed"),
        })

    edge_df = pd.DataFrame(edge_rows)

    display(edge_df[["length_m", "speed_kph", "travel_time_s"]].describe())

    print("Missing length:", edge_df["length_m"].isna().sum())
    print("Missing speed_kph:", edge_df["speed_kph"].isna().sum())
    print("Missing travel_time_s:", edge_df["travel_time_s"].isna().sum())
else:
    edge_df = None
    print("Chưa có graph để chạy edge QA.")


Chưa có graph để chạy edge QA.


In [8]:

if edge_df is not None:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.hist(edge_df["speed_kph"].dropna(), bins=30)
    ax.set_xlabel("speed_kph")
    ax.set_ylabel("Edge count")
    ax.set_title("OSM edge speed distribution")
    ax.grid(axis="y", alpha=0.25)
    plt.show()



### Cảnh báo về `speed_kph`

`add_edge_speeds` có thể phải **impute** speed cho edge thiếu `maxspeed`. Vì vậy:

- không gọi đây là tốc độ thực tế TP.HCM;
- không gọi `travel_time` là thời gian giao thông thực;
- dùng nó làm **free-flow / network baseline**;
- sau đó mới hiệu chỉnh bằng local traffic hoặc scenario multiplier.


## 7. Snap toàn bộ points vào OSM nodes — vectorized

In [9]:

POINT_NODE_MAP_PATH = OSM / "point_node_map.csv"

point_node_map = None

if G is not None:
    # nearest_nodes được vectorize: không loop 9,910 lần.
    node_ids, snap_dist = ox.distance.nearest_nodes(
        G,
        X=points["longitude"].to_numpy(),
        Y=points["latitude"].to_numpy(),
        return_dist=True,
    )

    point_node_map = points.copy()
    point_node_map["osmid"] = node_ids
    point_node_map["snap_distance_m"] = snap_dist
    point_node_map["snap_review_flag"] = (
        point_node_map["snap_distance_m"] > REVIEW_SNAP_DISTANCE_M
    )

    point_node_map.to_csv(POINT_NODE_MAP_PATH, index=False)
    print("Saved:", POINT_NODE_MAP_PATH)
    display(point_node_map.head())
else:
    print("Chưa có graph → chưa snap points.")


Chưa có graph → chưa snap points.


## 8. Snap-distance QA

In [10]:

if point_node_map is not None:
    snap_summary = (
        point_node_map.groupby("point_type")["snap_distance_m"]
        .agg(["count", "mean", "median", "max"])
        .reset_index()
    )

    snap_summary["review_count"] = (
        point_node_map.groupby("point_type")["snap_review_flag"].sum().values
    )

    display(snap_summary)

    snap_summary.to_csv(OSM / "snap_qa.csv", index=False)


In [11]:

if point_node_map is not None:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.hist(point_node_map["snap_distance_m"], bins=40)
    ax.axvline(REVIEW_SNAP_DISTANCE_M, linestyle="--", label="review threshold")
    ax.set_xlabel("Snap distance (m)")
    ax.set_ylabel("Point count")
    ax.set_title("Distance từ input coordinate đến nearest OSM node")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    plt.show()



### Gate

Nếu nhiều điểm có `snap_distance_m` lớn:

1. kiểm tra coordinate;
2. kiểm tra bbox có cắt road network;
3. kiểm tra point có nằm trong khu vực inaccessible/non-drive;
4. cân nhắc snap vào **edge** thay vì node ở phiên bản nâng cao;
5. không âm thầm giữ một point snap quá xa.


## 9. Map entity → node

In [12]:

if point_node_map is not None:
    incidents_nodes = (
        point_node_map[point_node_map["point_type"] == "incident"]
        [["entity_id", "scenario_id", "replication_id", "osmid", "snap_distance_m"]]
        .rename(columns={"entity_id": "incident_id", "osmid": "incident_node"})
    )

    hospital_nodes = (
        point_node_map[point_node_map["point_type"] == "hospital"]
        [["entity_id", "osmid", "snap_distance_m"]]
        .rename(columns={"entity_id": "hospital_id", "osmid": "hospital_node"})
    )

    charger_nodes = (
        point_node_map[point_node_map["point_type"] == "charger"]
        [["entity_id", "osmid", "snap_distance_m"]]
        .rename(columns={"entity_id": "charger_id", "osmid": "charger_node"})
    )

    fleet_nodes = (
        point_node_map[point_node_map["point_type"] == "fleet_initial"]
        [["entity_id", "scenario_id", "replication_id", "osmid", "snap_distance_m"]]
        .rename(columns={"entity_id": "vehicle_id", "osmid": "vehicle_node"})
    )

    print("incident nodes:", len(incidents_nodes))
    print("hospital nodes:", len(hospital_nodes))
    print("charger nodes:", len(charger_nodes))
    print("fleet nodes:", len(fleet_nodes))



## 10. OD Strategy

### Incident → Hospital

Có 9.000 × 6 = 54.000 cặp.

Không chạy 54.000 Dijkstra độc lập.

Với mỗi hospital target:

- reverse graph;
- chạy **một** single-source Dijkstra từ hospital;
- lookup tất cả incident nodes.

Như vậy chỉ cần khoảng **6 Dijkstra runs cho travel time** và 6 cho distance.

### Hospital → Charger

Chỉ 6 × 4 = 24 cặp.

### Fleet → Incident

Không build thành core static OD vì fleet location là **endogenous** và thay đổi theo B0/B3/B4/B6 trong rolling DES.

Dynamic layer nên cache theo key:

```text
(origin_node, destination_node, weight)
```


## 11. Build Incident → Hospital OD cache

In [13]:

INC_HOSP_OD_PATH = OSM / "incident_hospital_od.csv.gz"

incident_hospital_od = None

if G is not None and point_node_map is not None:
    G_rev = G.reverse(copy=False)

    rows = []

    for _, h in hospital_nodes.iterrows():
        h_id = h["hospital_id"]
        h_node = int(h["hospital_node"])

        # Trên reverse graph:
        # H -> I tương đương I -> H trên original directed graph.
        tt_to_h = nx.single_source_dijkstra_path_length(
            G_rev,
            h_node,
            weight="travel_time",
        )

        dist_to_h = nx.single_source_dijkstra_path_length(
            G_rev,
            h_node,
            weight="length",
        )

        for _, i in incidents_nodes.iterrows():
            i_node = int(i["incident_node"])
            tt = tt_to_h.get(i_node, np.nan)
            dist = dist_to_h.get(i_node, np.nan)

            rows.append({
                "scenario_id": i["scenario_id"],
                "replication_id": i["replication_id"],
                "incident_id": i["incident_id"],
                "incident_node": i_node,
                "hospital_id": h_id,
                "hospital_node": h_node,
                "freeflow_shortest_travel_time_s": tt,
                "network_shortest_distance_m": dist,
                "reachable": bool(np.isfinite(tt) and np.isfinite(dist)),
            })

    incident_hospital_od = pd.DataFrame(rows)
    incident_hospital_od.to_csv(
        INC_HOSP_OD_PATH,
        index=False,
        compression="gzip",
    )

    print("Saved:", INC_HOSP_OD_PATH)
    print("rows:", len(incident_hospital_od))
    print("reachable rate:", incident_hospital_od["reachable"].mean())
    display(incident_hospital_od.head())
else:
    print("Chưa có graph/point map → chưa build incident→hospital OD.")


Chưa có graph/point map → chưa build incident→hospital OD.



### Lưu ý phương pháp

Hai cột:

- `freeflow_shortest_travel_time_s`
- `network_shortest_distance_m`

được tối ưu **riêng theo hai edge weights khác nhau**, nên không nhất thiết mô tả cùng một route.

Đối với energy model chính thức, nếu muốn tính **distance của fastest route**, cần reconstruct route tối ưu theo `travel_time` rồi cộng `length` trên chính route đó. Notebook này tạo cache nền trước.


## 12. Build Hospital → Charger OD cache

In [14]:

HOSP_CHG_OD_PATH = OSM / "hospital_charger_od.csv"

hospital_charger_od = None

if G is not None and point_node_map is not None:
    rows = []

    charger_lookup = {
        row["charger_id"]: int(row["charger_node"])
        for _, row in charger_nodes.iterrows()
    }

    for _, h in hospital_nodes.iterrows():
        h_id = h["hospital_id"]
        h_node = int(h["hospital_node"])

        tt_from_h = nx.single_source_dijkstra_path_length(
            G,
            h_node,
            weight="travel_time",
        )

        dist_from_h = nx.single_source_dijkstra_path_length(
            G,
            h_node,
            weight="length",
        )

        for c_id, c_node in charger_lookup.items():
            tt = tt_from_h.get(c_node, np.nan)
            dist = dist_from_h.get(c_node, np.nan)

            rows.append({
                "hospital_id": h_id,
                "hospital_node": h_node,
                "charger_id": c_id,
                "charger_node": c_node,
                "freeflow_shortest_travel_time_s": tt,
                "network_shortest_distance_m": dist,
                "reachable": bool(np.isfinite(tt) and np.isfinite(dist)),
            })

    hospital_charger_od = pd.DataFrame(rows)
    hospital_charger_od.to_csv(HOSP_CHG_OD_PATH, index=False)

    print("Saved:", HOSP_CHG_OD_PATH)
    display(hospital_charger_od)
else:
    print("Chưa có graph/point map → chưa build hospital→charger OD.")


Chưa có graph/point map → chưa build hospital→charger OD.


## 13. Optional — Initial Fleet → Incident reference OD

In [15]:

INITIAL_FLEET_OD_PATH = OSM / "fleet_incident_initial_od.csv.gz"

if (
    BUILD_INITIAL_FLEET_INCIDENT_OD
    and G is not None
    and point_node_map is not None
):
    rows = []

    # Chỉ ghép vehicle và incident cùng scenario/replication.
    incidents_by_rep = {
        key: grp
        for key, grp in incidents_nodes.groupby(["scenario_id", "replication_id"])
    }

    for _, v in fleet_nodes.iterrows():
        key = (v["scenario_id"], v["replication_id"])
        target_incidents = incidents_by_rep.get(key)

        if target_incidents is None:
            continue

        v_node = int(v["vehicle_node"])

        tt_from_v = nx.single_source_dijkstra_path_length(
            G,
            v_node,
            weight="travel_time",
        )
        dist_from_v = nx.single_source_dijkstra_path_length(
            G,
            v_node,
            weight="length",
        )

        for _, i in target_incidents.iterrows():
            i_node = int(i["incident_node"])

            tt = tt_from_v.get(i_node, np.nan)
            dist = dist_from_v.get(i_node, np.nan)

            rows.append({
                "scenario_id": v["scenario_id"],
                "replication_id": v["replication_id"],
                "vehicle_id": v["vehicle_id"],
                "vehicle_node": v_node,
                "incident_id": i["incident_id"],
                "incident_node": i_node,
                "freeflow_shortest_travel_time_s": tt,
                "network_shortest_distance_m": dist,
                "reachable": bool(np.isfinite(tt) and np.isfinite(dist)),
                "reference_only": True,
            })

    fleet_incident_initial_od = pd.DataFrame(rows)
    fleet_incident_initial_od.to_csv(
        INITIAL_FLEET_OD_PATH,
        index=False,
        compression="gzip",
    )

    print("Saved:", INITIAL_FLEET_OD_PATH)
    print("rows:", len(fleet_incident_initial_od))
else:
    print(
        "Skipped initial fleet→incident OD. "
        "Đây là mặc định đúng cho rolling DES vì vehicle location sẽ thay đổi."
    )


Skipped initial fleet→incident OD. Đây là mặc định đúng cho rolling DES vì vehicle location sẽ thay đổi.


## 14. OD QA

In [16]:

if incident_hospital_od is not None:
    unreachable_ih = (~incident_hospital_od["reachable"]).sum()

    print("Incident→Hospital rows:", len(incident_hospital_od))
    print("Unreachable:", unreachable_ih)

    display(
        incident_hospital_od[
            ["freeflow_shortest_travel_time_s", "network_shortest_distance_m"]
        ].describe()
    )


In [17]:

if incident_hospital_od is not None:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.hist(
        incident_hospital_od.loc[
            incident_hospital_od["reachable"],
            "freeflow_shortest_travel_time_s",
        ] / 60,
        bins=40,
    )
    ax.set_xlabel("Free-flow shortest travel time (min)")
    ax.set_ylabel("OD pair count")
    ax.set_title("Incident → Hospital network travel time")
    ax.grid(axis="y", alpha=0.25)
    plt.show()


## 15. Network vs straight-line sanity check

In [18]:

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))


In [19]:

if incident_hospital_od is not None:
    inc_xy = (
        inc[["incident_id", "incident_lat", "incident_lon"]]
        .drop_duplicates("incident_id")
    )
    hosp_xy = hosp[["hospital_id", "latitude", "longitude"]].copy()

    qa = (
        incident_hospital_od
        .merge(inc_xy, on="incident_id", how="left")
        .merge(
            hosp_xy,
            on="hospital_id",
            how="left",
            suffixes=("_incident", "_hospital"),
        )
    )

    qa["haversine_m"] = haversine_m(
        qa["incident_lat"],
        qa["incident_lon"],
        qa["latitude"],
        qa["longitude"],
    )

    qa["network_to_haversine_ratio"] = (
        qa["network_shortest_distance_m"] / qa["haversine_m"]
    )

    display(
        qa.loc[
            qa["reachable"] & np.isfinite(qa["network_to_haversine_ratio"]),
            "network_to_haversine_ratio",
        ].describe()
    )


In [20]:

if incident_hospital_od is not None:
    ratio = qa.loc[
        qa["reachable"]
        & np.isfinite(qa["network_to_haversine_ratio"])
        & (qa["network_to_haversine_ratio"] < 10),
        "network_to_haversine_ratio",
    ]

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.hist(ratio, bins=40)
    ax.axvline(1.0, linestyle="--", label="network = straight-line")
    ax.set_xlabel("Network distance / Haversine distance")
    ax.set_ylabel("OD pair count")
    ax.set_title("Network detour ratio sanity check")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    plt.show()



### Interpretation

Thông thường network distance không nên nhỏ hơn Haversine một cách có hệ thống. Nếu xuất hiện nhiều ratio < 1:

- kiểm tra snap;
- kiểm tra duplicate/same node;
- kiểm tra coordinate;
- kiểm tra đơn vị.


## 16. Metadata + reproducibility record

In [21]:

NETWORK_METADATA_PATH = OSM / "network_metadata.json"

if G is not None:
    meta = {
        "osmnx_version": ox.__version__,
        "network_type": NETWORK_TYPE,
        "bbox_order": ["west", "south", "east", "north"],
        "bbox": {
            "west": WEST,
            "south": SOUTH,
            "east": EAST,
            "north": NORTH,
        },
        "graph_source": graph_source,
        "nodes": int(G.number_of_nodes()),
        "edges": int(G.number_of_edges()),
        "graph_crs": str(G.graph.get("crs")),
        "speed_model": "OSMnx add_edge_speeds default imputation; requires later local calibration",
        "travel_time_model": "edge length / speed_kph free-flow baseline",
        "traffic_applied": False,
        "emergency_vehicle_priority_modelled": False,
        "snap_review_threshold_m": REVIEW_SNAP_DISTANCE_M,
        "generated_files": [
            "study_graph.graphml",
            "point_node_map.csv",
            "snap_qa.csv",
            "incident_hospital_od.csv.gz",
            "hospital_charger_od.csv",
        ],
    }

    NETWORK_METADATA_PATH.write_text(
        json.dumps(meta, indent=2),
        encoding="utf-8",
    )

    print(json.dumps(meta, indent=2))
else:
    print("RUN_NETWORK=False → network_metadata chưa tạo.")


RUN_NETWORK=False → network_metadata chưa tạo.



# 17. Output kỳ vọng

Sau khi `RUN_NETWORK=True` và chạy notebook:

```text
data/osmnx/
├── study_graph.graphml
├── point_node_map.csv
├── snap_qa.csv
├── incident_hospital_od.csv.gz
├── hospital_charger_od.csv
├── network_metadata.json
└── fleet_incident_initial_od.csv.gz   # optional
```

## Gate trước Notebook 03 / DES

Chỉ chuyển tiếp nếu:

- [ ] graph tải thành công và lưu được;
- [ ] `speed_kph` không missing;
- [ ] `travel_time` không missing;
- [ ] point snapping hoàn tất;
- [ ] review các point có snap distance lớn;
- [ ] Incident→Hospital OD đạt reachability hợp lý;
- [ ] Hospital→Charger OD đạt reachability hợp lý;
- [ ] network/Haversine ratio không có pattern bất thường;
- [ ] metadata ghi rõ OSMnx version và bbox;
- [ ] chưa diễn giải free-flow OSMnx là traffic thực tế.

## Bước tiếp theo

Notebook 03 nên thực hiện:

```text
network free-flow OD
        ↓
time-dependent traffic multiplier
        ↓
policy-specific dynamic vehicle state
        ↓
candidate feasibility
        ↓
B0 / B3 / B4 / B6
```
